In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import nltk
from nltk.stem import WordNetLemmatizer
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

In [ ]:
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')

In [ ]:
lemmatizer=WordNetLemmatizer()
def clean(text):
    text=text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text=re.sub(r'[^\w\s]',"",text)
    words=text.split()
    words=[lemmatizer.lemmatize(word,pos='v')for word in words]
    return " ".join(words)

In [ ]:
ds = load_dataset("milistu/AMAZON-Products-2023")
df=pd.DataFrame(ds['train'].select_columns(['title', 'description', 'main_category']))
df['text']=(df['title']+' '+df['description'])

In [ ]:
df['text']=df['text'].apply(clean)
df.head()

In [ ]:
cmp_df=df[df['main_category'].notna()].copy()
msng_df=df[df['main_category'].isna()].copy()
cmp_df['main_category']=cmp_df['main_category'].str.lower()

In [ ]:
vc = cmp_df['main_category'].value_counts()
cmp_df['main_category'] = cmp_df['main_category'].apply(
    lambda x: x if vc[x] >= 50 else "other")
x_train,x_test,y_train,y_test=train_test_split(cmp_df['text'],cmp_df['main_category'],test_size=0.2,random_state=42,stratify=cmp_df['main_category'])

In [ ]:
lr_model=Pipeline([("cng_vctr",TfidfVectorizer(max_features=50000,ngram_range=(1,2),min_df=2,max_df=0.9,stop_words="english")),
                ('lr',LogisticRegression(max_iter=2000,class_weight="balanced"))])
nb_model=Pipeline([("cng_vctr",TfidfVectorizer(max_features=50000,ngram_range=(1,2),min_df=2,max_df=0.9,stop_words="english")),
                ('nb',MultinomialNB())])
svm_model=Pipeline([("cng_vctr",TfidfVectorizer(max_features=50000,ngram_range=(1,2),min_df=2,max_df=0.9,stop_words="english")),
                ('svm',LinearSVC(class_weight="balanced"))])

In [ ]:
lr_model.fit(x_train,y_train)
nb_model.fit(x_train,y_train)
svm_model.fit(x_train,y_train)

In [ ]:
lr_pred=lr_model.predict(x_test)
nb_pred=nb_model.predict(x_test)
svm_pred=svm_model.predict(x_test)

In [ ]:
print("...........lr_report............\n")
print(classification_report(y_test,lr_pred))


In [ ]:
print("\n\n..........nb_report...........\n")
print(classification_report(y_test,nb_pred))

In [ ]:
print("\n\n.........svm_report........\n")
print(classification_report(y_test,svm_pred))

In [ ]:
msng_df['text']=(msng_df['title']+' '+msng_df['description'])
msng_df['text']=msng_df['text'].apply(clean)

In [ ]:
msng_df['main_category']=svm_model.predict(msng_df['text'])

In [ ]:
msng_df['main_category'].value_counts()

In [ ]:
cmp_df=cmp_df.drop(['text'],axis=1)
msng_df=msng_df.drop(['text'],axis=1)

In [ ]:
cmp_df.head()

In [ ]:
msng_df.head()

In [ ]:
final_df=pd.concat([cmp_df,msng_df],axis=0)
print(final_df.isna().sum())
print(final_df.isnull().sum())

In [ ]:
final_df['title']=final_df['title'].apply(clean)
final_df['description']=final_df['description'].apply(clean)

In [ ]:
final_df.head()

In [ ]:
final_df.to_csv("clean_data.csv",index=False)